In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA



In [ ]:

# TASK 1: Load, Set Index, and Plot Raw Time Series Data
df = pd.read_csv('indian_city_daily_temperatures.csv')

# Crucial Time Series Step: Parse dates and establish a chronological datetime index
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

plt.figure(figsize=(12, 5))
plt.plot(df.index, df['Temperature_C'], color='tab:orange', alpha=0.8, label='Daily Temp')
plt.title('Daily Average Temperature Profile (Indian City)')
plt.xlabel('Timeline')
plt.ylabel('Temperature (°C)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.show()



In [ ]:


# TASK 2: Time Series Decomposition (Trend, Seasonal, Residual)
# Using period=365 since the primary seasonal pattern repeats annually
decomposition = seasonal_decompose(df['Temperature_C'], model='additive', period=365)

# Extract individual components
trend = decomposition.trend
seasonal = decomposition.seasonal
residual = decomposition.resid

# Plotting the separated sub-components
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

axes[0].plot(df.index, df['Temperature_C'], label='Original Data', color='black')
axes[0].legend(loc='upper left')
axes[0].set_title('Time Series Structural Decomposition')

axes[1].plot(df.index, trend, label='Trend Component', color='blue')
axes[1].legend(loc='upper left')

axes[2].plot(df.index, seasonal, label='Seasonal Component (Annual Cycle)', color='green')
axes[2].legend(loc='upper left')

axes[3].plot(df.index, residual, label='Residuals / Irregular Noise', color='red', linestyle='none', marker='.')
axes[3].legend(loc='upper left')

plt.tight_layout()
plt.show()



In [ ]:


# TASK 3: Augmented Dickey-Fuller (ADF) Stationarity Verification

adf_result = adfuller(df['Temperature_C'].dropna())

print(f"ADF Test Statistic : {adf_result[0]:.4f}")
print(f"p-value            : {adf_result[1]:.4e}")
print("\nCritical Values Framework:")
for key, value in adf_result[4].items():
    print(f"  {key}: {value:.4f}")

if adf_result[1] < 0.05:
    print("\nConclusion: The p-value is less than 0.05. We reject the null hypothesis.")
    print("The time series is STATIONARY (it lacks an explosive or dominant long-term unit root trend).")
else:
    print("\nConclusion: The p-value is greater than 0.05. We fail to reject the null hypothesis.")
    print("The time series is NON-STATIONARY (it requires differencing before modelling).")
print("\n" + "="*60 + "\n")



In [ ]:

# TASK 4: Moving Average Smoothing (Window Size = 7 Days)

# Computing a 7-day centered or rolling mean to strip out daily noise spikes
df['7Day_MA'] = df['Temperature_C'].rolling(window=7, center=True).mean()

plt.figure(figsize=(12, 5))
plt.plot(df.index, df['Temperature_C'], color='lightgray', label='Original Daily Data', alpha=0.7)
plt.plot(df.index, df['7Day_MA'], color='darkred', linewidth=2, label='7-Day Moving Average Smoothed')
plt.title('7-Day Moving Average Smoothing vs Original Observations')
plt.xlabel('Timeline')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



In [ ]:

# TASK 5: ARIMA Modeling and 7-Day Out-of-Sample Forecasting

# Based on the annual seasonality and our stationary test:
# p=2 (autoregressive term), d=0 (stationary, no differencing needed), q=2 (moving average term)
# Note: For strict annual macro cycles, SARIMA is preferred, but standard ARIMA(2,0,2) fits your baseline criteria here.
p, d, q = 2, 0, 2
arima_model = ARIMA(df['Temperature_C'], order=(p, d, q))
fitted_model = arima_model.fit()

# Forecast the next 7 steps out of sample (next 7 days after 2025-12-31)
forecast_steps = 7
forecast_res = fitted_model.get_forecast(steps=forecast_steps)
forecast_values = forecast_res.predicted_mean
forecast_dates = forecast_values.index

print("--- 7-Day Temperature Forecast Predictions ---")
for date, temp in zip(forecast_dates.strftime('%Y-%m-%d'), forecast_values):
    print(f"  Date: {date} -> Predicted Temperature: {temp:.2f}°C")

# Plot the last 60 days of original historical context data along with the new forecast
historical_window = df.tail(60)

plt.figure(figsize=(12, 5))
plt.plot(historical_window.index, historical_window['Temperature_C'], marker='o', color='blue', label='Historical Actuals (Last 60 Days)')
plt.plot(forecast_dates, forecast_values, marker='s', color='crimson', linestyle='--', linewidth=2, label='7-Day ARIMA Ahead Forecast')
plt.title('ARIMA(2,0,2) Out-of-Sample Temperature Forecast Line')
plt.xlabel('Timeline')
plt.ylabel('Temperature (°C)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()